In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# cmfe(기존 LTAU 기반)와 cue4unc(MC Dropout / Deep Ensemble)의 결과를 함께 비교한다.
# 세 결과 모두 동일한 CUE 아키텍처 + 동일한 5-fold 내부 CV 파이프라인에서 나온 것이며,
# 유일한 차이는 modality 불확실성을 계산하는 방법(u_m(x))이다:
#   - cmfe:            LTAU (loss-trajectory 기반 k-NN 불확실성)
#   - cue4unc/MC_Dropout:  MC Dropout (T회 확률적 forward pass의 예측 분산)
#   - cue4unc/Deep_Ensemble: Deep Ensemble (서로 다른 seed로 학습한 독립 모델들의 예측 분산)
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'
cue4unc_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue4unc/results'

datasets = ['DIS', 'INH']
# cue4unc의 run_cmfe_mc_dropout.sh / run_cmfe_ensemble.sh가 학습하는 타겟과 동일하게 맞춘다.
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# 비교할 세 가지 소스: (표시 이름, metrics.csv 루트 경로)
model_sources = {
    'CUE (LTAU)': os.path.join(cue_base, '{dataset}', '{target}', 'v1', 'CUE'),
    'CUE (MC Dropout)': os.path.join(cue4unc_base, 'MC_Dropout', '{dataset}', '{target}', 'v2', 'CUE'),
    'CUE (Deep Ensemble)': os.path.join(cue4unc_base, 'Deep_Ensemble', '{dataset}', '{target}', 'v1', 'CUE'),
}

# 그래프에 표시될 모델 순서
model_order = ['CUE (MC Dropout)', 'CUE (Deep Ensemble)', 'CUE (LTAU)']

In [2]:
def load_metric(model_dir_template, dataset, target):
    """cue/cue4unc의 metrics.csv(train.py::aggregate_metrics 출력)를 읽어
    model_name == 'mlp' 행의 rmse/r2/pcc(+ _std)를 뽑아온다.
    cmfe와 cue4unc 모두 동일한 aggregate_metrics 함수를 사용하므로 스키마가 동일하다."""
    metrics_path = os.path.join(model_dir_template.format(dataset=dataset, target=target), 'metrics.csv')
    if not os.path.exists(metrics_path):
        return None

    df_m = pd.read_csv(metrics_path)
    mlp_row = df_m[df_m['model_name'] == 'mlp']
    if mlp_row.empty:
        return None

    values = {}
    for col in mlp_row.columns:
        if col in ['model_name', 'Unnamed: 0']:
            continue
        norm_col = col.lower()
        if norm_col in metrics_of_interest or norm_col.replace('_std', '') in metrics_of_interest:
            values[norm_col] = mlp_row[col].values[0]
    return values if values else None


raw_rows = []
for d_name in datasets:
    for target in targets:
        for model_name, dir_template in model_sources.items():
            vals = load_metric(dir_template, d_name, target)
            if vals is not None:
                raw_rows.append({'Dataset': d_name, 'Target': target, 'Model': model_name, **vals})

df_total = pd.DataFrame(raw_rows)
print(f"✅ 데이터 로드 완료: {len(df_total)}개의 항목이 수집되었습니다.")

✅ 데이터 로드 완료: 24개의 항목이 수집되었습니다.


In [3]:
# rmse는 낮을수록, r2/pcc는 높을수록 좋은 성능이므로 순위 방향을 다르게 매긴다 (1등이 가장 좋은 성능).
LOWER_IS_BETTER = {'rmse': True, 'r2': False, 'pcc': False}


def build_ranked_table(df_all, dataset, target, metrics=metrics_of_interest):
    """지정한 dataset/target에 대해 Model별 metric 값 + 순위를 담은 표를 만든다.

    Args:
        df_all (pd.DataFrame): load_metric()으로 수집한 전체 결과(df_total).
        dataset (str): 'DIS' | 'INH'.
        target (str): 'PI' | 'Ssel' | 'WS2' | 'RS2'.
        metrics (list[str]): 값/순위를 매길 metric 목록.

    Returns:
        Optional[pd.DataFrame]: 컬럼이 Model, {metric}, {metric}_rank ... 순으로
            반복되는 표. 해당 dataset/target에 데이터가 하나도 없으면 None.
    """
    df_sub = df_all[(df_all['Dataset'] == dataset) & (df_all['Target'] == target)]
    df_sub = df_sub.set_index('Model').reindex(model_order)
    df_sub = df_sub.dropna(subset=[m for m in metrics if m in df_sub.columns], how='all')
    if df_sub.empty:
        return None

    table = pd.DataFrame(index=df_sub.index)
    for m in metrics:
        if m not in df_sub.columns:
            continue
        table[m] = df_sub[m]
        table[f'{m}_rank'] = df_sub[m].rank(ascending=LOWER_IS_BETTER.get(m, True), method='min')

    return table.reset_index()


ranked_tables = {}
for d_name in datasets:
    for target in targets:
        table = build_ranked_table(df_total, d_name, target)
        if table is None:
            continue
        ranked_tables[(d_name, target)] = table
        print(f"\n=== [{d_name} / {target}] ===")
        print(table.to_string(index=False))


=== [Kd / PI] ===
               Model     rmse  rmse_rank       r2  r2_rank      pcc  pcc_rank
   CMFE (MC Dropout) 0.241046        1.0 0.128918      1.0 0.388636       1.0
CMFE (Deep Ensemble) 0.242486        3.0 0.118858      3.0 0.373700       2.0
         CMFE (LTAU) 0.241524        2.0 0.125334      2.0 0.365724       3.0

=== [Kd / Ssel] ===
               Model     rmse  rmse_rank       r2  r2_rank      pcc  pcc_rank
   CMFE (MC Dropout) 0.903339        2.0 0.048102      2.0 0.301681       2.0
CMFE (Deep Ensemble) 0.907471        3.0 0.040212      3.0 0.298106       3.0
         CMFE (LTAU) 0.894523        1.0 0.065169      1.0 0.314014       1.0

=== [Kd / WS(alpha=2)] ===
               Model     rmse  rmse_rank       r2  r2_rank      pcc  pcc_rank
   CMFE (MC Dropout) 0.283473        2.0 0.209875      2.0 0.481319       2.0
CMFE (Deep Ensemble) 0.286180        3.0 0.193923      3.0 0.467453       3.0
         CMFE (LTAU) 0.281553        1.0 0.220999      1.0 0.490397       

In [4]:
def plot_comparison_styled(df_plot, dataset, target_name, metric_name, output_dir):
    """지표 라벨은 지우되, Y축 눈금 수치와 배경은 유지한 버전.
    LTAU/MC Dropout/Deep Ensemble을 해치 없이 colors 색상으로만 구분한다.
    막대 높이도 텍스트 라벨과 동일하게 소수점 셋째 자리까지 반올림한 값을 사용한다."""
    df_target = df_plot.set_index('Model').reindex(model_order).dropna(subset=[metric_name]).reset_index()
    if df_target.empty: return

    models = df_target['Model'].tolist()
    # 텍스트 라벨(f'{height:.3f}')과 막대 높이를 일치시키기 위해 셋째 자리까지 반올림
    values = [round(v, 3) for v in df_target[metric_name].tolist()]
    colors = ['#ef476f', '#ffd166', '#06d6a0']

    y_min, y_max = min(values), max(values)
    margin = (y_max - y_min) * 0.45 if (y_max - y_min) != 0 else 0.1

    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(len(models))

    bars = ax.bar(x, values, color=colors[:len(models)], edgecolor='black', alpha=0.9, linewidth=1)

    for i, bar in enumerate(bars):
        height = bar.get_height()
        # 막대 위 성능 수치 표시 (크기 15, 세로형)
        ax.text(bar.get_x() + bar.get_width()/2., height + (margin * 0.03),
                f'{height:.3f}', ha='center', va='bottom', fontsize=15,
                fontweight='bold', rotation=90)

    # --- 텍스트 라벨 제거 ---
    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels([]) # X축 모델명만 숨김

    # --- Y축 수치 및 배경 유지 ---
    ax.tick_params(axis='y', labelsize=11) # Y축 숫자 크기 조절
    ax.tick_params(axis='x', which='both', length=0) # X축 눈금선만 깔끔하게 제거

    ax.grid(axis='y', linestyle='--', alpha=0.5)
    for spine in ax.spines.values():
        spine.set_visible(True)

    # 상단 여백 확보
    ax.set_ylim(max(0, y_min - (margin*0.1)), y_max + margin)

    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, f"{target_name}_{metric_name}.png")
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"✅ 저장 완료: {save_path}")

In [5]:
plot_base_dir = os.path.join(cue_base, 'plots', 'uncertainty_comparison')

for d_name in datasets:
    for target in targets:
        df_target = df_total[(df_total['Dataset'] == d_name) & (df_total['Target'] == target)]
        if df_target.empty: continue

        g_name = 'DIS' if d_name == 'DIS' else 'INH'
        output_path = os.path.join(plot_base_dir, g_name)
        for m in metrics_of_interest:
            if m in df_target.columns:
                plot_comparison_styled(df_target, d_name, target, m, output_path)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/PI_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/PI_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/PI_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/Ssel_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/Ssel_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/Ssel_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/WS(alpha=2)_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/WS(alpha=2)_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/WS(alpha=2)_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/RS(k=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/RS(k=2)_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/DIS/RS(k=2)_pcc.png


... (3 outputs left out)

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/INH/Ssel_pcc.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/INH/WS(alpha=2)_rmse.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/INH/WS(alpha=2)_r2.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/INH/WS(alpha=2)_pcc.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/INH/RS(k=2)_rmse.png
✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/INH/RS(k=2)_r2.png


✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/plots/uncertainty_comparison/INH/RS(k=2)_pcc.png
